# Notebook 5: RAG - PDF Document Search

## Learning Objectives:
- Load and process PDF documents
- Create vector databases
- Implement semantic search
- Integrate RAG with chatbot

In [ ]:
%pip install langchain langchain-community langgraph python-dotenv openpyxl pandas langchain-openai google-search-results pypdf faiss-cpu --quiet

In [1]:
from langchain_core.tools import tool
from langchain_community.utilities import GoogleSerperAPIWrapper
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from typing import Literal

from ai_course_utils import load_llm_from_env, load_use_case_config, load_pdf_for_rag

In [2]:
use_case_file = "../data/use_case_Movie_Recommendation.xlsx"
config = load_use_case_config(use_case_file)
system_prompt = config.get('agent_prompt')

✓ Use case loaded: ../data/use_case_Movie_Recommendation.xlsx
  Components: user, agent_prompt, url


## Load PDF Document

**USER INPUT**: Provide path to your PDF file

In [3]:
# ============================================================================
# USER INPUT: Specify PDF path
# ============================================================================
pdf_file = "../data/The_98th Academy Awards_2026.pdf"

# Load and process PDF
documents, vectorstore, retriever = load_pdf_for_rag(pdf_file)

print(f'\n✓ PDF processed')
print(f'  Pages: {len(documents)}')
print(f'  Ready for semantic search')

📄 Loading PDF: ../data/The_98th Academy Awards_2026.pdf
  ✓ Loaded 11 pages
  ✓ Created 14 chunks
  ✓ Vector store created
  ✓ Retriever ready (top 3 results)

✓ PDF processed
  Pages: 11
  Ready for semantic search


## Test Document Retrieval

In [4]:
# Test the retriever directly
test_query = 'Who is nominated for Best Actor?'
results = retriever.invoke(test_query)

print(f'Query: {test_query}\n')
for i, doc in enumerate(results, 1):
    print(f'Result {i}:')
    print(doc.page_content[:200])
    print()

Query: Who is nominated for Best Actor?

Result 1:
SELECT A CATEGORY
ACTOR IN A LEADING ROLE
NOMINEES
TIMOTHÉE CHALAMET
Marty Supreme
LEONARDO DICAPRIO
One Battle after Another
ETHAN HAWKE
Blue Moon
MICHAEL B. JORDAN
Sinners
WAGNER MOURA
The Secret Ag

Result 2:
COSTUME DESIGN
NOMINEES
AVATAR: FIRE AND ASH
Deborah L. Scott
FRANKENSTEIN
Kate Hawley
HAMNET
Malgosia Turzanska
MARTY SUPREME
Miyako Bellizzi
SINNERS
Ruth E. Carter
DIRECTING
NOMINEES
HAMNET
Chloé Zh

Result 3:
Nominees to be determined
THE PERFECT NEIGHBOR
Geeta Gandbhir, Alisa Payne, Nikon Kwantu and Sam Bisbee
DOCUMENTARY SHORT FILM
NOMINEES
ALL THE EMPTY ROOMS
Joshua Seftel and Conall Jones
ARMED ONLY WI



## Define Tools with Document Search

In [8]:
@tool
def search_web(query: str) -> str:
    """Search web for current info."""
    search = GoogleSerperAPIWrapper()
    return search.run(query)

@tool
def search_documents(query: str) -> str:
    """Search the loaded PDF document.
    
    Use this for:
    - Questions about Oscar nominations
    - Information in the uploaded PDF
    - Specific movie/actor details from document
    """
    docs = retriever.invoke(query)
    results = []
    for doc in docs:
        results.append(doc.page_content)
    return '\n\n---\n\n'.join(results)

tools = [search_web, search_documents]
print(f'✓ Tools: {[t.name for t in tools]}')

✓ Tools: ['search_web', 'search_documents']


In [9]:
llm = load_llm_from_env()
llm_with_tools = llm.bind_tools(tools)

🤖 Loading LLM: openai / gpt-4.1-mini


In [10]:
def assistant(state):
    return {'messages': [llm_with_tools.invoke([SystemMessage(content=system_prompt)] + state['messages'])]}

def should_continue(state):
    return 'tools' if state['messages'][-1].tool_calls else '__end__'

graph_builder = StateGraph(MessagesState)
graph_builder.add_node('assistant', assistant)
graph_builder.add_node('tools', ToolNode(tools))
graph_builder.add_edge(START, 'assistant')
graph_builder.add_conditional_edges('assistant', should_continue)
graph_builder.add_edge('tools', 'assistant')
graph = graph_builder.compile(checkpointer=MemorySaver())

print('✓ Graph with RAG built')

✓ Graph with RAG built


In [11]:
def chat(msg, tid='default'):
    for event in graph.stream({'messages': [HumanMessage(content=msg)]}, {'configurable': {'thread_id': tid}}, stream_mode='values'):
        pass
    return event['messages'][-1].content

print('🎬 RAG-enabled chatbot ready!')

🎬 RAG-enabled chatbot ready!


## Test RAG Capability

In [12]:
# Query about PDF content
query = 'Who is nominated for Best Picture?'
print(f'User: {query}')
response = chat(query)
print(f'Bot: {response}')

User: Who is nominated for Best Picture?
Bot: The nominees for Best Picture have not been determined yet according to the information available. Would you like to know about nominees in other categories or from a specific year?


In [13]:
# Follow-up question
query2 = 'Tell me more about Marty Supreme'
print(f'\nUser: {query2}')
response2 = chat(query2)
print(f'Bot: {response2}')


User: Tell me more about Marty Supreme
Bot: "Marty Supreme" appears to be a notable film with multiple nominations. It is nominated in categories such as Film Editing, Costume Design, Directing (Josh Safdie), and Writing (Original Screenplay by Ronald Bronstein & Josh Safdie). 

Would you like to know more about the plot, cast, or reviews of "Marty Supreme"? Or are you interested in other films by the same directors?


## Summary

### What We Built:
✅ PDF document loading and processing  
✅ Vector database for semantic search  
✅ Document search tool  
✅ Combined web search + document search  

### Key Concepts:
- **RAG**: Retrieval-Augmented Generation
- **Vector Store**: FAISS database for similarity search
- **Embeddings**: Text converted to numerical vectors
- **Semantic Search**: Find by meaning, not keywords

### Next: Notebook 6 - Taxonomy Classification